In [1]:
import os

## Write ```Dockerfile``` to local drive

In [2]:
%%writefile Dockerfile

FROM python:3.9
    
# update package manager
RUN apt-get update

# update pip
RUN pip install --upgrade pip

# copy requirements
COPY requirements.txt .

# install dependencies
RUN pip install -r requirements.txt

# copy script into container
COPY script.py .

# run script when image is run
CMD ["python3", "script.py"]

Writing Dockerfile


## Write ```requirements.txt```

In [3]:
%%writefile requirements.txt

pyarrow==9.0.0
fsspec==2022.10.0
s3fs==2022.10.0

pandas==1.2.4
boto3==1.24.59
shap==0.40.0
matplotlib==3.6.1
catboost==1.0.4

Writing requirements.txt


## Write ```script.py``` to local drive

In [4]:
%%writefile script.py

import os
import pandas as pd
import boto3
import pickle
import shap
import matplotlib.pyplot as plt

# download from s3
def download_from_s3(str_local_path, str_bucket_path, str_project):
    # download file
    boto3.client('s3').download_file(str_project, str_bucket_path, str_local_path)

# upload to s3
def upload_to_s3(str_local_path, str_bucket_path, str_project):
    boto3.resource('s3').Bucket(str_project).Object(str_bucket_path).upload_file(str_local_path)

# constants
str_project = '20231010-gen-xii'
str_target = 'target'
str_dirname_output = './output'
  
# make output dir
try:
    os.mkdir(str_dirname_output)
except FileExistsError:
    pass

###############################################################################
# HYPERPARAMETERS
###############################################################################
# get df_hyperparameters
str_filename = 'df_hyperparameters.csv'
str_uri = f's3://{str_project}/02_pricing_pd/02_model/02_model/12_step_function/{str_filename}'
df = pd.read_csv(str_uri)
# convert to dict
dict_hyperparameters = dict(zip(df['keys'], df['values']))

# get filename for valid
str_filename_valid = dict_hyperparameters['STR_FILENAME_VALID']
print(f'Valid filename: {str_filename_valid}')

##################################################################################

# import model
print('Importing best model...')
str_filename = 'df_tuning.csv'
str_uri = f's3://{str_project}/02_pricing_pd/02_model/02_model/03_lambda_concat_tuning/{str_filename}'
df = pd.read_csv(str_uri)
# get iteration
int_best_iteration = df['iteration'].iloc[0]

# get model
str_filename = f'dict_model_inference_{int_best_iteration}.pkl'
str_local_path = f'{str_dirname_output}/{str_filename}'
str_bucket_path = f'02_pricing_pd/02_model/02_model/02_batch_tuning/models/{str_filename}'
download_from_s3(
    str_local_path=str_local_path, 
    str_bucket_path=str_bucket_path, 
    str_project=str_project,
)
dict_pipeline = pickle.load(open(str_local_path, 'rb'))
cls_model_inference = dict_pipeline['model_inference']
list_cols_model = list(cls_model_inference.feature_names_)

# initialize explainer
print('Initializing tree explainer class...')
cls_explainer = shap.TreeExplainer(
    cls_model_inference,
)

# read validation data
print('Importing validation data...')
str_uri = f's3://{str_project}/02_pricing_pd/02_model/00_preprocessing/02_make_dfs/{str_filename_valid}'
df = pd.read_parquet(str_uri, columns=list_cols_model)

# get shap vals
print('Getting SHAP values...')
arr_shap_values = cls_explainer.shap_values(
    df,
)
# make df shap
df_shap = pd.DataFrame(arr_shap_values, columns=list_cols_model)

# iterate
print('Creating plots...')
for col in list_cols_model:
    # ax
    fig, ax = plt.subplots(figsize=(9,5))
    ax.set_title(col) # title
    ax.set_xlabel(col) # xlabel
    ax.set_ylabel('SHAP') # y
    # if numeric
    if df[col].dtype in ['float64','int64']:
        # convert to z-score
        flt_mean = df[col].mean()
        flt_std = df[col].std()
        df['zscore'] = (df[col] - flt_mean) / flt_std
        # make into absolute value
        df['zscore_abs'] = df['zscore'].abs()
        # assign to df_shap
        df_shap['zscore_abs'] = list(df['zscore_abs'])
        # subset < 3 because z score cutoff is 3
        df_tmp = df[df['zscore_abs'] < 3].copy()
        df_shap_tmp = df_shap[df_shap['zscore_abs'] < 3].copy()
        # save series
        ser_col = df_tmp[col]
        ser_shap = df_shap_tmp[col]
    else:
        # save series
        ser_col = df[col]
        ser_shap = df_shap[col]
    # plot
    ax.scatter(ser_col, ser_shap) # plot
    plt.tight_layout() # fix overlap
    str_filename = f'{col}.png'
    str_local_path = f'{str_dirname_output}/{str_filename}'
    plt.savefig(str_local_path, bbox_inches='tight')
    # upload to s3
    str_bucket_path = f'02_pricing_pd/02_model/02_model/08_batch_pd_plots_valid/plots/{str_filename}'
    upload_to_s3(
        str_local_path=str_local_path, 
        str_bucket_path=str_bucket_path,
        str_project=str_project,
    )
    # close
    plt.close()

Writing script.py


## Build and push to ECR

In [5]:
%%sh

# image name
image=genxii-pd-plots-valid

# Get the account number associated with the current IAM credentials
account=$(aws sts get-caller-identity --query Account --output text)

# did we have an error?
if [ $? -ne 0 ]
then
    exit 255
fi

# Get the region defined in the current configuration (default to us-west-2 if none defined)
region=$(aws configure get region)
region=${region:-us-west-2}

# get destination of repo
fullname="${account}.dkr.ecr.${region}.amazonaws.com/${image}:latest"

# If the repository doesn't exist in ECR, create it.
aws ecr describe-repositories --repository-names "${image}" > /dev/null 2>&1

# if it doesnt exist...create it
if [ $? -ne 0 ]
then
    aws ecr create-repository --repository-name "${image}" > /dev/null
fi

# Get the login command from ECR and execute it directly
aws ecr get-login-password --region "${region}" | docker login --username AWS --password-stdin "${account}".dkr.ecr."${region}".amazonaws.com

# Build the docker image locally with the image name and then push it to ECR
# with the full name.

# build and add tag
docker build  -t ${image} .
docker tag ${image} ${fullname}
# push to ecr
docker push ${fullname}

WARNING! Your password will be stored unencrypted in /home/ec2-user/.docker/config.json.
Configure a credential helper to remove this warning. See
https://docs.docker.com/engine/reference/commandline/login/#credentials-store



Login Succeeded
Sending build context to Docker daemon  36.35kB
Step 1/7 : FROM python:3.9
 ---> fc0d8a3ea4c2
Step 2/7 : RUN apt-get update
 ---> Using cache
 ---> 31350654e329
Step 3/7 : RUN pip install --upgrade pip
 ---> Using cache
 ---> fa3d5a6126e9
Step 4/7 : COPY requirements.txt .
 ---> Using cache
 ---> 44d8d5193977
Step 5/7 : RUN pip install -r requirements.txt
 ---> Running in e1a16081ade6
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 48.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.8/138.8 kB 17.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 93.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.5/132.5 kB 18.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 567.6/567.6 kB 48.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.8/11.8 MB 93.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.2/76.2 MB 33.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 269.4/269.4 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.5/61.5 kB 953.1 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 311.0/311.0 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 38.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 66.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.0/53.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 43.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.1/103.1 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 502.5/502.5 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.8/79.8

## Clean-up

In [6]:
# rm files
for str_file in ['Dockerfile','requirements.txt','script.py']:
    try:
        os.remove(f'./{str_file}')
    except:
        pass